# Consumer Feedback & Unseen-Template Evaluation

**Pregunta:** ¿qué tan bien generaliza un clasificador entrenado con textos generados?

**SIMULACIÓN independiente:** 650 reseñas bilingües creadas con plantillas, semilla 2240. Las etiquetas no provienen de anotación humana. No se unen estas reseñas a facturas UCI ni se inventa una relación entre sentimiento y ventas.

## 1. Preparación

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from a checkout of retail-analytics-pipeline.")
sys.path.insert(0, str(ROOT / "src"))
REPORTS = ROOT / "reports"


**Dónde:** Jupyter en Codespaces o una copia del repositorio. La carpeta raíz se identifica por `pyproject.toml`; los mismos módulos se usan desde `main.py`.

## 2. Crear y revisar feedback

In [2]:
from market_insights.data import simulate_feedback
feedback = simulate_feedback()
display(feedback.sentiment_label.value_counts())
display(feedback[['review_text','sentiment_label','template_id']].head())

sentiment_label
positive    273
negative    215
neutral     162
Name: count, dtype: int64
                                         review_text  ...                                  template_id
0  La calidad es buena y el precio razonable. Lo ...  ...   La calidad es buena y el precio razonable.
1  El empaque falló y el contenido decepciona. I ...  ...  El empaque falló y el contenido decepciona.
2  No me gustó y no lo recomiendo. Lo probé duran...  ...              No me gustó y no lo recomiendo.
3  Me encantó, cumplió mis expectativas. Lo usé d...  ...        Me encantó, cumplió mis expectativas.
4  It's ok, nothing special. Lo encontré en el su...  ...                    It's ok, nothing special.

[5 rows x 3 columns]


**Stack:** NumPy y pandas. Se conservan 273 positive, 215 negative y 162 neutral. Cada texto completo es único, pero comparte frases base con otras reseñas. `template_id` permite detectar esa dependencia.

## 3. Limpieza y evaluación

In [3]:
from market_insights.sentiment import clean_text, evaluate_sentiment
print(clean_text('No, no me gustó!'))
metrics, heldout, matrix, model = evaluate_sentiment(feedback)
display(metrics.round(3))

no no me gustó
               split            model  ...  f1_negative  f1_neutral
0  random_stratified  logreg_unigrams  ...        0.916       0.896
1  random_stratified   logreg_bigrams  ...        0.914       0.923
2  random_stratified      naive_bayes  ...        0.914       0.938
3   unseen_templates  logreg_unigrams  ...        0.026       0.038
4   unseen_templates   logreg_bigrams  ...        0.024       0.038
5   unseen_templates      naive_bayes  ...        0.025       0.038

[6 rows x 9 columns]


**Stack:** TF-IDF, Logistic Regression y MultinomialNB. Se conservan acentos y negación. El vocabulario se aprende solo con train. Se comparan un split estratificado y un split por familias de plantillas: una frase base de test no aparece en train.

## 4. Matriz de confusión y errores

In [4]:
from market_insights.visualization import plot_sentiment
plot_sentiment(metrics, matrix, REPORTS/'figures')
display(heldout.loc[heldout.sentiment_label.ne(heldout.predicted)].head(8))

   feedback_id  ... predicted
4      FB-0005  ...  positive
13     FB-0014  ...  negative
23     FB-0024  ...  negative
26     FB-0027  ...  negative
31     FB-0032  ...  positive
36     FB-0037  ...  negative
41     FB-0042  ...  negative
45     FB-0046  ...  positive

[8 rows x 5 columns]


**Stack:** pandas y Matplotlib. La matriz corresponde a Logistic Regression con bigramas sobre familias no vistas. El modelo de entrega está fijado, no elegido por el mejor test. Revisar F1 por clase y ejemplos; no considerar la accuracy aleatoria como prueba de rendimiento en reseñas reales.

![Figura de la ejecución registrada](../reports/figures/sentiment_confusion.png)

## 5. Model card y próximo experimento

In [5]:
display(metrics.loc[metrics.model.eq('logreg_bigrams'), ['split','accuracy','macro_f1','f1_positive','f1_negative','f1_neutral']])

               split  accuracy  macro_f1  f1_positive  f1_negative  f1_neutral
1  random_stratified  0.923077  0.922161     0.929825     0.913580    0.923077
4   unseen_templates  0.197279  0.135313     0.343949     0.023529    0.038462


El buen resultado del split aleatorio se reduce mucho con familias no vistas. Próximo paso: etiquetas humanas y texto real de una fuente con permiso de uso, además de validación por idioma y dominio. No se presenta esta simulación como un clasificador listo para producción.